# BookLM-RAG: от двух поисков до проверяемой цитаты

**Вопрос опыта:** можно ли без сети и готовой RAG-библиотеки построить тракт, в котором фильтр доступа применяется до оценки, BM25 и семантический поиск совершают разные ошибки, RRF объединяет кандидатов, граф добавляет связанный факт, а каждое утверждение ответа ссылается на реально переданный фрагмент?

**Ожидаемый результат:** публичный пользователь находит точный бюджет KV-кэша BookLM, не видит закрытый документ, а манифест `booklm-rag-v1` фиксирует корпус, конфигурацию и измерения. Разделы 1–10 используют только стандартную библиотеку Python; полный листинг главы в разделе 11 добавляет NumPy. Ноутбук выполняется после Restart Kernel → Run All.

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict, deque
from dataclasses import dataclass, asdict
from hashlib import sha256
from math import log, sqrt
from pathlib import Path
import json
import platform
import re
import sys

SEED = 11  # Алгоритмы детерминированы; поле оставлено в паспорте опыта.
print({"python": sys.version.split()[0], "platform": platform.platform(), "seed": SEED})


## 1. Корпус и происхождение

Каждая запись имеет версию, источник, уровень доступа и родителя. `chunk_id` зависит от этих полей и текста: изменение источника создаёт новую идентичность вместо незаметного перемещения старой цитаты. Закрытый фрагмент намеренно содержит очень привлекательные слова запроса — позже проверим, что он даже не получает поисковую оценку для публичного субъекта.

In [ ]:
@dataclass(frozen=True)
class Chunk:
    document_id: str
    version: str
    parent_id: str
    source: str
    page: int
    access: str
    text: str
    entities: tuple[str, ...]
    chunk_id: str = ""


def stable_chunk_id(chunk: Chunk) -> str:
    payload = "\x1f".join((
        chunk.document_id, chunk.version, chunk.parent_id,
        chunk.source, str(chunk.page), chunk.text.strip(),
    ))
    return "ch-" + sha256(payload.encode("utf-8")).hexdigest()[:12]


def make_chunk(**kwargs) -> Chunk:
    draft = Chunk(**kwargs)
    return Chunk(**{**asdict(draft), "chunk_id": stable_chunk_id(draft)})


chunks = [
    make_chunk(
        document_id="booklm-contract", version="1", parent_id="booklm",
        source="BOOKLM.md#parameters", page=1, access="public",
        text=("BookLM содержит 6 622 464 параметра. В BF16 одни параметры "
              "занимают 13 244 928 байт, то есть примерно 12,63 МБ."),
        entities=("booklm", "bf16"),
    ),
    make_chunk(
        document_id="booklm-contract", version="1", parent_id="booklm",
        source="BOOKLM.md#kv-cache", page=2, access="public",
        text=("KV-кэш BookLM хранит ключи и значения. Для шести блоков это "
              "1536 байт на токен, а для контекста из 512 токенов — "
              "786 432 байта, или 768 КБ."),
        entities=("booklm", "kv-cache"),
    ),
    make_chunk(
        document_id="serving-guide", version="3", parent_id="serving",
        source="serving.md#vllm", page=4, access="public",
        text=("vLLM обслуживает несколько запросов, использует непрерывное "
              "формирование пакета и страничное размещение KV-кэша."),
        entities=("vllm", "kv-cache"),
    ),
    make_chunk(
        document_id="deployment-registry", version="17", parent_id="deployment",
        source="registry://deploy-17", page=1, access="public",
        text=("Развёртывание deploy-17 запускает артефакт booklm-rag-v1 "
              "на ускорителе H100-80GB. Ёмкость HBM равна 80 ГБ."),
        entities=("deploy-17", "booklm-rag-v1", "h100-80gb"),
    ),
    make_chunk(
        document_id="deployment-registry", version="9", parent_id="deployment",
        source="registry://deploy-09", page=1, access="public",
        text=("Архивное развёртывание deploy-09 запускало BookLM на A100-40GB. "
              "Эта запись не описывает действующий сервис."),
        entities=("deploy-09", "booklm", "a100-40gb"),
    ),
    make_chunk(
        document_id="customer-secret", version="2", parent_id="finance",
        source="vault://finance/customer-secret", page=1, access="finance",
        text=("Секретный лимит клиента равен 900 000 токенов. "
              "KV-кэш BookLM 512 токенов, память, H100 и deploy-17."),
        entities=("customer-x", "booklm"),
    ),
]

assert len({c.chunk_id for c in chunks}) == len(chunks)
assert stable_chunk_id(chunks[0]) == chunks[0].chunk_id
[(c.chunk_id, c.source, c.access) for c in chunks]


## 2. Фильтр доступа перед оценкой

В учебном стенде политика проста: `public` видят все, `finance` — только роль finance. В реальной системе это может быть предикат арендатора, группы, грифа и времени. Важно место применения: сначала строим разрешённое представление корпуса, затем передаём его любому поисковому модулю.

In [ ]:
def visible_chunks(subject_roles: set[str], corpus: list[Chunk]) -> list[Chunk]:
    return [
        chunk for chunk in corpus
        if chunk.access == "public" or chunk.access in subject_roles
    ]


public_corpus = visible_chunks({"public"}, chunks)
finance_corpus = visible_chunks({"public", "finance"}, chunks)
secret_id = next(c.chunk_id for c in chunks if c.access == "finance")
assert secret_id not in {c.chunk_id for c in public_corpus}
assert secret_id in {c.chunk_id for c in finance_corpus}
assert len(public_corpus) == 5 and len(finance_corpus) == 6


## 3. Разреженный поиск BM25

Токенизатор сохраняет дефисы в идентификаторах. Реализация ниже намеренно мала, но содержит оба существенных механизма BM25: насыщение частоты и нормировку длины.

In [ ]:
TOKEN_RE = re.compile(r"[0-9a-zа-яё]+(?:-[0-9a-zа-яё]+)*", re.IGNORECASE)


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text.lower())


class BM25:
    def __init__(self, corpus: list[Chunk], k1: float = 1.5, b: float = 0.75):
        self.corpus = list(corpus)
        self.k1 = k1
        self.b = b
        self.term_counts = [Counter(tokenize(c.text)) for c in corpus]
        self.lengths = [sum(counts.values()) for counts in self.term_counts]
        self.avg_length = sum(self.lengths) / max(1, len(self.lengths))
        self.df = Counter()
        for counts in self.term_counts:
            self.df.update(counts.keys())

    def idf(self, term: str) -> float:
        n_docs = len(self.corpus)
        doc_freq = self.df.get(term, 0)
        return log(1.0 + (n_docs - doc_freq + 0.5) / (doc_freq + 0.5))

    def score(self, query: str, index: int) -> float:
        counts = self.term_counts[index]
        length = self.lengths[index]
        total = 0.0
        for term in set(tokenize(query)):
            frequency = counts.get(term, 0)
            if not frequency:
                continue
            norm = self.k1 * (1.0 - self.b + self.b * length / self.avg_length)
            total += self.idf(term) * frequency * (self.k1 + 1.0) / (frequency + norm)
        return total

    def search(self, query: str, top_k: int = 5) -> list[tuple[Chunk, float]]:
        scored = [(chunk, self.score(query, i)) for i, chunk in enumerate(self.corpus)]
        scored = [item for item in scored if item[1] > 0.0]
        return sorted(scored, key=lambda item: (-item[1], item[0].chunk_id))[:top_k]


bm25 = BM25(public_corpus)
exact_results = bm25.search("KV-кэш BookLM 512 токенов", top_k=3)
assert exact_results[0][0].source == "BOOKLM.md#kv-cache"
[(c.source, round(score, 4)) for c, score in exact_results]


## 4. Учебный плотный поиск

Настоящий dense retriever получает представления из обученного трансформера. Здесь внешняя модель сделала бы ноутбук сетевым и невоспроизводимым, поэтому семантическую часть заменяет объявленный словарь понятий. Он не претендует на качество эмбеддера: его роль — показать контракт `encode → normalize → exact cosine`, с которым затем сравнивают ANN.

In [ ]:
CONCEPTS = {
    "booklm": {"booklm", "booklm-rag-v1"},
    "kv": {"kv-кэш", "kv", "кэш", "кеша", "кеш", "ключи", "ключей", "значения", "значений"},
    "memory": {"память", "памяти", "объём", "объем", "байт", "байта", "кб", "гб", "hbm"},
    "tokens": {"токен", "токена", "токенов", "позиция", "позиций", "контекст", "контекста"},
    "deployment": {"развёртывание", "развертывание", "deploy-17", "сервис", "запускает", "работает"},
    "gpu": {"gpu", "ускоритель", "ускорителе", "h100-80gb", "a100-40gb"},
}
CONCEPT_NAMES = tuple(CONCEPTS)


def semantic_vector(text: str) -> tuple[float, ...]:
    terms = set(tokenize(text))
    raw = [float(len(terms & CONCEPTS[name])) for name in CONCEPT_NAMES]
    norm = sqrt(sum(value * value for value in raw))
    return tuple(value / norm for value in raw) if norm else tuple(raw)


def cosine(a: tuple[float, ...], b: tuple[float, ...]) -> float:
    assert len(a) == len(b)
    return sum(x * y for x, y in zip(a, b))


class ExactDenseIndex:
    def __init__(self, corpus: list[Chunk]):
        self.corpus = list(corpus)
        self.vectors = [semantic_vector(c.text) for c in corpus]

    def search(self, query: str, top_k: int = 5) -> list[tuple[Chunk, float]]:
        query_vector = semantic_vector(query)
        scored = [(c, cosine(query_vector, v)) for c, v in zip(self.corpus, self.vectors)]
        scored = [item for item in scored if item[1] > 0.0]
        return sorted(scored, key=lambda item: (-item[1], item[0].chunk_id))[:top_k]


dense = ExactDenseIndex(public_corpus)
semantic_query = "Какой объём памяти занимает кеш ключей и значений для 512 позиций?"
dense_results = dense.search(semantic_query, top_k=3)
assert dense_results[0][0].source == "BOOKLM.md#kv-cache"
[(c.source, round(score, 4)) for c, score in dense_results]


## 5. RRF и объяснимое переранжирование

BM25 и cosine имеют несопоставимые шкалы. Reciprocal Rank Fusion использует места. После него учебный reranker добавляет только наблюдаемые признаки: покрытие термов, сущность и штраф архивной записи. В промышленной системе эту функцию заменяет cross-encoder, но вход и журнал остаются теми же.

In [ ]:
def reciprocal_rank_fusion(
    rankings: list[list[tuple[Chunk, float]]], constant: int = 60
) -> list[tuple[Chunk, float]]:
    fused = defaultdict(float)
    by_id = {}
    for ranking in rankings:
        for rank, (chunk, _score) in enumerate(ranking, start=1):
            fused[chunk.chunk_id] += 1.0 / (constant + rank)
            by_id[chunk.chunk_id] = chunk
    return sorted(
        ((by_id[chunk_id], score) for chunk_id, score in fused.items()),
        key=lambda item: (-item[1], item[0].chunk_id),
    )


def rerank(query: str, candidates: list[tuple[Chunk, float]]) -> list[tuple[Chunk, float]]:
    query_terms = set(tokenize(query))
    scored = []
    for chunk, rrf_score in candidates:
        terms = set(tokenize(chunk.text))
        coverage = len(query_terms & terms) / max(1, len(query_terms))
        concept_overlap = cosine(semantic_vector(query), semantic_vector(chunk.text))
        archive_penalty = 0.20 if "архивное" in terms else 0.0
        score = 10.0 * rrf_score + coverage + concept_overlap - archive_penalty
        scored.append((chunk, score))
    return sorted(scored, key=lambda item: (-item[1], item[0].chunk_id))


def hybrid_search(query: str, corpus: list[Chunk], top_k: int = 5) -> list[tuple[Chunk, float]]:
    sparse = BM25(corpus).search(query, top_k=max(10, top_k))
    semantic = ExactDenseIndex(corpus).search(query, top_k=max(10, top_k))
    return rerank(query, reciprocal_rank_fusion([sparse, semantic]))[:top_k]


hybrid_results = hybrid_search(semantic_query, public_corpus, top_k=3)
assert hybrid_results[0][0].source == "BOOKLM.md#kv-cache"
assert secret_id not in {c.chunk_id for c, _ in hybrid_results}
[(c.source, round(score, 4)) for c, score in hybrid_results]


## 6. Граф добавляет связь, а не правдоподобный текст

Рёбра ниже имеют собственные источники. Поисковый кандидат даёт начальные сущности, после чего ограниченный BFS добавляет только доступные фрагменты, подтверждающие соседние отношения. Это минимальная версия seed → graph expansion → source chunks, а не полноценная онтология.

In [ ]:
@dataclass(frozen=True)
class Edge:
    subject: str
    predicate: str
    object: str
    source_chunk_id: str


by_source = {c.source: c for c in chunks}
edges = [
    Edge("deploy-17", "runs", "booklm-rag-v1", by_source["registry://deploy-17"].chunk_id),
    Edge("deploy-17", "uses_gpu", "h100-80gb", by_source["registry://deploy-17"].chunk_id),
    Edge("h100-80gb", "memory_bytes", "85899345920", by_source["registry://deploy-17"].chunk_id),
    Edge("booklm-rag-v1", "parent", "booklm", by_source["BOOKLM.md#parameters"].chunk_id),
    Edge("booklm", "kv_bytes_at_512", "786432", by_source["BOOKLM.md#kv-cache"].chunk_id),
]


def graph_expand(seed_entities: set[str], allowed: list[Chunk], max_hops: int = 2) -> list[Chunk]:
    allowed_ids = {c.chunk_id for c in allowed}
    chunk_by_id = {c.chunk_id: c for c in allowed}
    adjacency = defaultdict(list)
    for edge in edges:
        adjacency[edge.subject].append(edge)
        adjacency[edge.object].append(edge)
    queue = deque((entity, 0) for entity in sorted(seed_entities))
    visited = set(seed_entities)
    evidence_ids = set()
    while queue:
        entity, depth = queue.popleft()
        if depth >= max_hops:
            continue
        for edge in adjacency[entity]:
            if edge.source_chunk_id in allowed_ids:
                evidence_ids.add(edge.source_chunk_id)
            for neighbor in (edge.subject, edge.object):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append((neighbor, depth + 1))
    return [chunk_by_id[cid] for cid in sorted(evidence_ids)]


expanded = graph_expand({"booklm-rag-v1"}, public_corpus, max_hops=2)
expanded_sources = {c.source for c in expanded}
assert "registry://deploy-17" in expanded_sources
assert "BOOKLM.md#parameters" in expanded_sources
assert secret_id not in {c.chunk_id for c in expanded}
sorted(expanded_sources)


## 7. Бюджет контекста и цитаты

Сборщик получает уже разрешённые и ранжированные фрагменты. Он дедуплицирует ID и никогда не обрезает источник так, чтобы цитата указывала на другой текст. Учебный генератор не является LLM: он извлекает число регулярным выражением, чтобы отдельно проверить механическую цепочку evidence → claim → citation.

In [ ]:
def pack_context(ranked: list[tuple[Chunk, float]], token_budget: int) -> list[Chunk]:
    packed = []
    seen = set()
    used = 0
    for chunk, _score in ranked:
        if chunk.chunk_id in seen:
            continue
        length = len(tokenize(chunk.text))
        if used + length > token_budget:
            continue
        packed.append(chunk)
        seen.add(chunk.chunk_id)
        used += length
    return packed


def answer_kv_question(packed: list[Chunk]) -> list[dict]:
    for chunk in packed:
        match = re.search(r"512 токенов[^.]*?([0-9 ]+) байта[^.]*?([0-9 ]+) КБ", chunk.text)
        if match:
            byte_count = match.group(1).replace(" ", "")
            kib = match.group(2).replace(" ", "")
            return [{
                "text": f"KV-кэш BookLM для 512 токенов занимает {byte_count} байта, или {kib} КБ.",
                "citations": [chunk.chunk_id],
            }]
    return [{"text": "В переданных источниках недостаточно данных.", "citations": []}]


def validate_citations(claims: list[dict], packed: list[Chunk]) -> None:
    packed_by_id = {c.chunk_id: c for c in packed}
    for claim in claims:
        assert claim["citations"], "проверяемое утверждение осталось без источника"
        for chunk_id in claim["citations"]:
            assert chunk_id in packed_by_id, "цитата не была передана генератору"
            evidence = packed_by_id[chunk_id].text
            assert "512 токенов" in evidence and "768 КБ" in evidence


ranked = hybrid_search("Сколько памяти занимает KV-кэш BookLM для 512 токенов?", public_corpus)
packed = pack_context(ranked, token_budget=70)
claims = answer_kv_question(packed)
validate_citations(claims, packed)
assert claims[0]["text"].endswith("768 КБ.")
claims


## 8. Раздельная оценка поиска

Gold хранит ID допустимого источника, а не готовую фразу ответа. Проверяем Recall@3 и reciprocal rank. Последний вопрос намеренно неответим публичным корпусом: закрытый фрагмент не должен улучшать метрику или попадать в трассу.

In [ ]:
gold = [
    ("KV-кэш BookLM 512 токенов", by_source["BOOKLM.md#kv-cache"].chunk_id),
    ("Объём кеша ключей и значений для 512 позиций", by_source["BOOKLM.md#kv-cache"].chunk_id),
    ("На каком GPU работает booklm-rag-v1?", by_source["registry://deploy-17"].chunk_id),
]


def evaluate_retrieval(corpus: list[Chunk], cases: list[tuple[str, str]], k: int = 3) -> dict:
    hits = 0
    reciprocal_ranks = []
    traces = []
    for query, expected_id in cases:
        results = hybrid_search(query, corpus, top_k=k)
        ids = [chunk.chunk_id for chunk, _score in results]
        hits += int(expected_id in ids)
        rank = ids.index(expected_id) + 1 if expected_id in ids else None
        reciprocal_ranks.append(1.0 / rank if rank else 0.0)
        traces.append({"query": query, "ids": ids, "expected": expected_id})
    return {
        f"recall_at_{k}": hits / len(cases),
        "mrr": sum(reciprocal_ranks) / len(cases),
        "traces": traces,
    }


metrics = evaluate_retrieval(public_corpus, gold, k=3)
assert metrics["recall_at_3"] == 1.0
assert all(secret_id not in trace["ids"] for trace in metrics["traces"])
unanswerable = hybrid_search("Каков секретный лимит клиента?", public_corpus, top_k=3)
assert all(chunk.document_id != "customer-secret" for chunk, _ in unanswerable)
metrics


## 9. Намеренно сломанная вариация

Замените в `hybrid_search` переданный `corpus` на глобальный `chunks`, выполните поиск, а фильтр доступа примените только к готовому top-3. Закрытый документ специально насыщен словами запроса и вытеснит разрешённый кандидат. Объясните две ошибки: утечку в поисковую трассу и падение полноты после поздней фильтрации. Затем восстановите исходный порядок и добавьте assert, что закрытый ID никогда не передавался BM25 или dense index.

## 10. Манифест `booklm-rag-v1`

Веса BookLM не менялись. Производный артефакт фиксирует родителя, корпус, алгоритмы, контроль доступа и фактические метрики. Файл мал и пригоден для сравнения двух запусков.

In [ ]:
def corpus_digest(corpus: list[Chunk]) -> str:
    records = [asdict(chunk) for chunk in sorted(corpus, key=lambda c: c.chunk_id)]
    payload = json.dumps(records, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return sha256(payload.encode("utf-8")).hexdigest()


manifest = {
    "id": "booklm-rag-v1",
    # веса родителя ноутбук не загружает, поэтому его SHA-256 не записан
    "parents": {"booklm-instruct-lora-v1": None},
    "created_by": "notebooks/12_rag.ipynb",
    "format": "hybrid-rag-index-spec",
    "format_version": 1,
    "corpus_sha256": corpus_digest(chunks),
    "public_view_sha256": corpus_digest(public_corpus),
    "index": {
        "sparse": {"algorithm": "BM25", "k1": 1.5, "b": 0.75},
        "dense": {"algorithm": "educational-exact-cosine", "dimensions": len(CONCEPT_NAMES)},
        "fusion": {"algorithm": "RRF", "constant": 60},
        "graph": {"algorithm": "bounded-BFS", "max_hops": 2},
    },
    "controls": {
        "access_filter_before_scoring": True,
        "citations_must_reference_packed_context": True,
        "network_required": False,
    },
    "metrics": {
        "recall_at_3": metrics["recall_at_3"],
        "mrr": metrics["mrr"],
        "citation_contract_passed": True,
        "forbidden_chunks_in_public_trace": 0,
    },
    "environment": {"python": sys.version.split()[0]},
}

output_path = Path("notebooks/artifacts/booklm-rag-v1/manifest.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
assert json.loads(output_path.read_text(encoding="utf-8"))["corpus_sha256"] == manifest["corpus_sha256"]
print(output_path, manifest["corpus_sha256"][:16], manifest["metrics"])


## 11. Полный листинг главы из `labs/ch12/`

В разделе «Сквозной пример BookLM-RAG» глава показывает только ключевые куски: BM25 по инвертированному
индексу, косинус, RRF, MMR, `retrieve` и `validate_citations`. Целиком этот путь лежит в файле
`labs/ch12/booklm_rag_pipeline.py`: шесть фрагментов о BookLM, фильтр прав до поиска, BM25, косинус по
учебным векторам, RRF, MMR, упаковка в бюджет, промпт с блоками EVIDENCE, ответ со ссылками `[S:chunk_id]`,
проверка цитат и метрики поиска. Файл самодостаточен, его можно запустить и отдельно:
`python labs/ch12/booklm_rag_pipeline.py` из корня репозитория.

Корпус в файле свой, проще, чем в разделах 1–10: права там заданы группами (`all`, `hr`), а векторы —
тремя осями «память», «устройство модели», «железо». Поэтому функция `run_lab` загружает файл через
`importlib` как отдельный модуль. Его имена (`chunks`, `bm25`, `rrf`) живут внутри модуля и не перетирают
одноимённые объекты этого ноутбука. Если не сработает хоть один `assert` файла, упадёт эта ячейка.


In [ ]:
import importlib.util


def find_labs():
    for base in (Path("labs/ch12"), Path("../labs/ch12")):
        if (base / "booklm_rag_pipeline.py").is_file():
            return base.resolve()
    raise FileNotFoundError("нет каталога labs/ch12: запускайте ноутбук из корня репозитория "
                            "или из каталога notebooks/")


LABS = find_labs()


def run_lab(name):
    """Выполняет labs/ch12/<name>.py заново как отдельный модуль и возвращает его."""
    spec = importlib.util.spec_from_file_location(f"labs_ch12_{name}", LABS / f"{name}.py")
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module      # свежий модуль на каждый запуск, без старого состояния
    spec.loader.exec_module(module)      # здесь выполняются все assert файла
    return module


lab = run_lab("booklm_rag_pipeline")

# Нужный фрагмент про 768 КБ стоит в контексте первым, закрытый HR-документ туда не попал.
assert lab.packed[0] == "booklm-md-v3-0042" and "768 КБ" in lab.prompt
assert "hr-salary-v1-0001" not in lab.packed
assert "hr-salary-v1-0001" in lab.packed_hr
# Тот же бюджет KV-кэша, что нашёл гибридный поиск в разделах 1–7 этого ноутбука.
assert "786 432 байта" in by_source["BOOKLM.md#kv-cache"].text and "786432 байта" in lab.DOCS["booklm-md-v3-0042"]
print("контекст:", lab.packed, f"({lab.used} слов)")


## Связь с промышленной системой

`BM25` соответствует инвертированному индексу; `ExactDenseIndex` — эталонному полному перебору, относительно которого проверяют HNSW/IVF; `reciprocal_rank_fusion` — объединителю кандидатов; `rerank` — месту cross-encoder; `graph_expand` — seed-and-expand GraphRAG; `pack_context` — токенному бюджету; `validate_citations` — первому уровню claim-to-source проверки. Замена любого компонента допустима, если сохраняются фильтр доступа, стабильные ID, трасса и тесты.